# 01 · Preprocessing — MTPL / ОГПО hackathon (leak-free)

Builds a **policy-level** modeling dataset from raw driver-level CSVs using Polars.

- Input:
  - `final_dataset/train.csv`
  - `final_dataset/test_final.csv`
- Output (only):
  - `outputs/preprocessed/train.csv`
  - `outputs/preprocessed/test.csv`

**Grain:** one row per `contract_number`.

**Targets (train only):** `is_claim`, `claim_amount`, `claim_cnt`.

**Leakage rules baked in:**
- No `_train_only` features.
- No features derived from any target column (`is_claim`, `claim_amount`, `claim_cnt`).
- No driver/car/insurer "claim rate", "avg_claim_amount", "sum_claim_amount" history.
- Structural aggregates only (counts, n_unique, stats over non-target columns).


## 1. Setup

In [ ]:
from __future__ import annotations

from pathlib import Path
import numpy as np
import polars as pl

# Locate repo root regardless of whether the notebook is launched from
# repo root or from a notebooks/ subfolder.
HERE = Path.cwd()
if (HERE / "final_dataset").exists():
    REPO_ROOT = HERE
elif (HERE.parent / "final_dataset").exists():
    REPO_ROOT = HERE.parent
else:
    raise FileNotFoundError(
        f"Cannot locate repo root containing final_dataset/ from cwd={HERE}"
    )

DATA_DIR = REPO_ROOT / "final_dataset"
PREP_DIR = REPO_ROOT / "outputs" / "preprocessed"
PREP_DIR.mkdir(parents=True, exist_ok=True)

# --- column groups ---------------------------------------------------------
TARGET_COLS = ["is_claim", "claim_amount", "claim_cnt"]
POLICY_FIRST_COLS = ["premium", "premium_wo_term"] + TARGET_COLS  # take .first() per contract

# Raw IDs / PII / near-constant fields to drop from the final modeling table.
RAW_DROP_COLS = [
    "driver_iin",
    "insurer_iin",
    "car_number",
    "unique_id",
    "is_individual_person",
    "is_individual_person_name",
    "is_residence",
    "is_residence_name",
]

# Names that are forbidden in the final feature set (leakage red flags).
FORBIDDEN_FEATURE_SUBSTRINGS = [
    "train_only",
    "claim_rate",
    "avg_claim_amount",
    "sum_claim_amount",
    "worst_driver_claim_rate",
]

# Official bonus-malus coefficient map (provided by the hackathon).
BM_COEF_MAP = {
    "M": 2.45, "0": 2.30, "1": 1.55, "2": 1.40, "3": 1.00,
    "4": 0.95, "5": 0.90, "6": 0.85, "7": 0.80, "8": 0.75,
    "9": 0.70, "10": 0.65, "11": 0.60, "12": 0.55, "13": 0.50,
}
BAD_BM_CLASSES = ["M", "0", "1", "2"]

# Hard validity windows for car_year / engine_*; quantile clipping is applied on top.
CAR_YEAR_MIN, CAR_YEAR_MAX = 1900, 2022
ENGINE_POWER_MIN, ENGINE_POWER_MAX = 20.0, 1000.0
ENGINE_VOLUME_MIN, ENGINE_VOLUME_MAX = 100.0, 10000.0

# Rare-bucket threshold for mark/model grouping (fit on TRAIN, applied to TEST).
RARE_THRESHOLD = 20

print(f"REPO_ROOT: {REPO_ROOT}")
print(f"DATA_DIR:  {DATA_DIR}")
print(f"PREP_DIR:  {PREP_DIR}")
print(f"polars:    {pl.__version__}")


## 2. Load CSVs

Strip the UTF-8 BOM from any column name (e.g. `\ufeff` in front of `unique_id`).


In [ ]:
def load_csv(path: Path) -> pl.DataFrame:
    df = pl.read_csv(
        path,
        infer_schema_length=10_000,
        ignore_errors=True,
        try_parse_dates=False,
        null_values=["", "NA", "NaN", "null", "NULL"],
    )
    rename = {c: c.lstrip("\ufeff") for c in df.columns if c.startswith("\ufeff")}
    if rename:
        df = df.rename(rename)
    return df


train_df = load_csv(DATA_DIR / "train.csv")
test_df  = load_csv(DATA_DIR / "test_final.csv")

print(f"train: {train_df.shape}")
print(f"test:  {test_df.shape}")
print(f"train cols (first 10): {train_df.columns[:10]}")

# Safety: required grain key must be present in both.
assert "contract_number" in train_df.columns, "train.csv is missing contract_number"
assert "contract_number" in test_df.columns,  "test_final.csv is missing contract_number"

# Safety: train must have all targets, test must NOT.
for t in TARGET_COLS:
    assert t in train_df.columns, f"train.csv is missing target column {t!r}"
test_target_cols_present = [t for t in TARGET_COLS if t in test_df.columns]
if test_target_cols_present:
    print(f"WARNING: test has target columns {test_target_cols_present} — will drop later.")


## 3. Driver-level cleaning

Adds **new** columns; never overwrites raw ones. No feature uses any target column.


In [ ]:
def compute_quantile(df: pl.DataFrame, col: str, q: float) -> float | None:
    """Quantile of a column robust to non-numeric dtypes / missing column."""
    if col not in df.columns:
        return None
    try:
        v = df.select(pl.col(col).cast(pl.Float64, strict=False).quantile(q)).item()
        return None if v is None else float(v)
    except Exception:
        return None


def clean_drivers(df: pl.DataFrame, is_train: bool) -> pl.DataFrame:
    """Driver-row level cleaning. Returns df with new feature columns added.

    IMPORTANT: nothing here is derived from is_claim / claim_amount / claim_cnt.
    """
    exprs: list[pl.Expr] = []

    # --- 3.1 operation_date -> year/month/quarter/weekday + missing flag -----
    if "operation_date" in df.columns:
        s = pl.col("operation_date").cast(pl.Utf8, strict=False)
        parsed = s.str.to_date(strict=False)
        exprs.extend([
            parsed.dt.year().alias("operation_year"),
            parsed.dt.month().alias("operation_month"),
            parsed.dt.quarter().alias("operation_quarter"),
            parsed.dt.weekday().alias("operation_dayofweek"),
            pl.col("operation_date").is_null().alias("operation_date_missing"),
        ])

    # --- 3.2 bonus_malus -> coefficient + flags ------------------------------
    if "bonus_malus" in df.columns:
        bm_str = pl.col("bonus_malus").cast(pl.Utf8, strict=False).str.strip_chars()
        mapped = bm_str.replace_strict(BM_COEF_MAP, default=None, return_dtype=pl.Float64)
        exprs.extend([
            mapped.alias("bonus_malus_coef"),
            bm_str.is_null().alias("bonus_malus_missing"),
            (bm_str.is_not_null() & mapped.is_null()).alias("bonus_malus_unmapped"),
            bm_str.is_in(BAD_BM_CLASSES).alias("bonus_malus_is_bad_class"),
            bm_str.cast(pl.Float64, strict=False).alias("bonus_malus_class_num"),
        ])

    # --- 3.3 car_year integer + validity window ------------------------------
    if "car_year" in df.columns:
        cy = pl.col("car_year").cast(pl.Float64, strict=False)
        cy_int = cy == cy.floor()
        cy_valid = (cy > CAR_YEAR_MIN) & (cy < CAR_YEAR_MAX) & cy_int
        exprs.extend([
            pl.when(cy_valid).then(cy).otherwise(None).alias("car_year_clean"),
            (~cy_valid).alias("car_year_outlier"),
            pl.col("car_year").is_null().alias("car_year_missing"),
        ])

    # --- 3.4 engine_power / engine_volume: clip with quantiles ---------------
    engine_rules = {
        "engine_power":  (ENGINE_POWER_MIN,  ENGINE_POWER_MAX),
        "engine_volume": (ENGINE_VOLUME_MIN, ENGINE_VOLUME_MAX),
    }
    for col, (lo, hi) in engine_rules.items():
        if col not in df.columns:
            continue
        raw = pl.col(col).cast(pl.Float64, strict=False)
        valid = raw.is_not_null() & (raw >= lo) & (raw <= hi)
        clean = pl.when(valid).then(raw).otherwise(None)
        p01 = compute_quantile(df, col, 0.01)
        p99 = compute_quantile(df, col, 0.99)
        exprs.append(raw.is_null().alias(f"{col}_missing"))
        exprs.append((raw.is_not_null() & ~valid).alias(f"{col}_outlier"))
        if p01 is not None and p99 is not None:
            clip_lo = max(p01, lo)
            clip_hi = min(p99, hi)
            exprs.append(clean.clip(clip_lo, clip_hi).alias(f"{col}_clipped"))
        else:
            exprs.append(clean.alias(f"{col}_clipped"))

    # --- 3.5 mark / model normalization (grouping fit on train below) --------
    for col in ["mark", "model"]:
        if col in df.columns:
            norm = (
                pl.col(col).cast(pl.Utf8, strict=False)
                  .str.strip_chars()
                  .str.to_uppercase()
                  .str.replace_all(r"\s+", " ")
            )
            exprs.append(
                pl.when(norm.is_null() | (norm == ""))
                  .then(pl.lit("UNKNOWN"))
                  .otherwise(norm)
                  .alias(f"{col}_clean")
            )

    df = df.with_columns(exprs)

    # --- 3.6 car_age category text -> numeric age + binary -------------------
    # (data has "до 7 лет включ." / "свыше 7 лет"). Not target-derived.
    if "car_age" in df.columns:
        ca_str = pl.col("car_age").cast(pl.Utf8, strict=False).str.strip_chars().str.to_lowercase()
        le7 = ca_str.str.contains(r"до\s*7|включ", strict=False)
        gt7 = ca_str.str.contains(r"свыше\s*7|старше\s*7", strict=False)
        df = df.with_columns([
            le7.alias("car_age_le_7"),
            gt7.alias("car_age_gt_7"),
            pl.when(le7).then(pl.lit(0))
              .when(gt7).then(pl.lit(1))
              .otherwise(None).alias("car_age_binary"),
            pl.when(le7).then(pl.lit("le_7"))
              .when(gt7).then(pl.lit("gt_7"))
              .otherwise(pl.lit("UNKNOWN")).alias("car_age_category"),
        ])

    # --- 3.7 numeric vehicle age from car_year_clean -------------------------
    if "car_year_clean" in df.columns:
        if "operation_year" in df.columns:
            age = pl.col("operation_year").cast(pl.Float64, strict=False) - pl.col("car_year_clean")
        else:
            age = pl.lit(float(CAR_YEAR_MAX)) - pl.col("car_year_clean")
        df = df.with_columns([
            age.alias("vehicle_age_from_year"),
            (age <= 2).alias("vehicle_age_is_new"),
            (age >= 15).alias("vehicle_age_is_old"),
        ])

    # --- 3.8 Cast all SCORE_* columns to Float64 -----------------------------
    score_cols = [c for c in df.columns if c.startswith("SCORE_")]
    if score_cols:
        df = df.with_columns([pl.col(c).cast(pl.Float64, strict=False) for c in score_cols])

    return df


def fit_apply_rare_groups(
    train: pl.DataFrame,
    test: pl.DataFrame,
    cols=("mark", "model"),
    rare_threshold: int = RARE_THRESHOLD,
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Fit rare-level grouping on TRAIN only, apply same map to TEST.

    Uses driver-row frequency of the *_clean column. Categories below
    threshold (or unseen in train) are bucketed as __RARE_OR_UNSEEN__.
    No target columns are used.
    """
    for col in cols:
        clean_col = f"{col}_clean"
        if clean_col not in train.columns:
            continue
        freq_col   = f"{col}_frequency_train"
        rare_col   = f"{col}_is_rare"
        grouped_col = f"{col}_grouped"

        # Drop pre-existing versions in case of re-run.
        for c in [freq_col, rare_col, grouped_col]:
            if c in train.columns:
                train = train.drop(c)
            if c in test.columns:
                test = test.drop(c)

        freq = train.group_by(clean_col).len().rename({"len": freq_col})

        for name, frame in [("train", train), ("test", test)]:
            frame = frame.join(freq, on=clean_col, how="left").with_columns([
                pl.col(freq_col).fill_null(0).alias(freq_col),
                (pl.col(freq_col).fill_null(0) < rare_threshold).alias(rare_col),
                pl.when(pl.col(freq_col).fill_null(0) < rare_threshold)
                  .then(pl.lit("__RARE_OR_UNSEEN__"))
                  .otherwise(pl.col(clean_col))
                  .alias(grouped_col),
            ])
            if name == "train":
                train = frame
            else:
                test = frame
    return train, test


train_clean = clean_drivers(train_df, is_train=True)
test_clean  = clean_drivers(test_df,  is_train=False)
train_clean, test_clean = fit_apply_rare_groups(train_clean, test_clean)

print(f"train_clean: {train_clean.shape}")
print(f"test_clean:  {test_clean.shape}")


## 4. Aggregate to policy level

One `.group_by("contract_number").agg(...)` per dataset. Premiums and targets are
duplicated across driver rows for the same policy — we take `.first()`.

No target-based aggregates. No `_train_only` columns.


In [ ]:
def aggregate_to_policy(df: pl.DataFrame, is_train: bool) -> pl.DataFrame:
    agg: list[pl.Expr] = []

    # --- 4.1 policy-level money + targets ------------------------------------
    # Premiums repeat per driver but belong to the policy -> .first() is correct.
    # Targets in TRAIN also repeat per driver -> .first().
    for c in POLICY_FIRST_COLS:
        if c in df.columns:
            agg.append(pl.col(c).cast(pl.Float64, strict=False).first().alias(c))

    # --- 4.2 structural counts (no targets used) -----------------------------
    agg.append(pl.len().alias("row_count_in_policy"))
    if "driver_iin"  in df.columns:
        agg.append(pl.col("driver_iin").n_unique().alias("n_unique_drivers"))
    if "car_number"  in df.columns:
        agg.append(pl.col("car_number").n_unique().alias("n_unique_cars"))
    if "insurer_iin" in df.columns:
        agg.append(pl.col("insurer_iin").n_unique().alias("n_unique_insurers"))

    # --- 4.3 driver risk stats -----------------------------------------------
    if "experience_year" in df.columns:
        ex = pl.col("experience_year").cast(pl.Float64, strict=False)
        agg.extend([
            ex.min().alias("min_experience_year"),
            ex.max().alias("max_experience_year"),
            ex.mean().alias("avg_experience_year"),
            ex.std().alias("std_experience_year"),
        ])

    if "bonus_malus_coef" in df.columns:
        bm = pl.col("bonus_malus_coef")
        agg.extend([
            bm.min().alias("min_bonus_malus_coef"),
            bm.max().alias("max_bonus_malus_coef"),
            bm.mean().alias("avg_bonus_malus_coef"),
            bm.std().alias("std_bonus_malus_coef"),
        ])
    if "bonus_malus_class_num" in df.columns:
        bmn = pl.col("bonus_malus_class_num")
        agg.extend([
            bmn.min().alias("min_bonus_malus_class_num"),
            bmn.max().alias("max_bonus_malus_class_num"),
            bmn.mean().alias("avg_bonus_malus_class_num"),
        ])
    if "bonus_malus_is_bad_class" in df.columns:
        agg.append(pl.col("bonus_malus_is_bad_class").cast(pl.Int8).max().alias("has_bad_bm_driver"))
    if "bonus_malus_unmapped" in df.columns:
        agg.append(pl.col("bonus_malus_unmapped").cast(pl.Int8).max().alias("has_unmapped_bm_class"))
    if "bonus_malus_missing" in df.columns:
        agg.append(pl.col("bonus_malus_missing").cast(pl.Int8).max().alias("has_missing_bm_driver"))

    # --- 4.4 vehicle stats ---------------------------------------------------
    if "car_year_clean" in df.columns:
        cy = pl.col("car_year_clean")
        agg.extend([
            cy.min().alias("min_car_year_clean"),
            cy.max().alias("max_car_year_clean"),
            cy.mean().alias("avg_car_year_clean"),
        ])
    if "car_age_binary" in df.columns:
        cab = pl.col("car_age_binary").cast(pl.Float64, strict=False)
        agg.extend([
            cab.min().alias("min_car_age_binary"),
            cab.max().alias("max_car_age_binary"),
            cab.mean().alias("avg_car_age_binary"),
        ])
    if "vehicle_age_from_year" in df.columns:
        vay = pl.col("vehicle_age_from_year").cast(pl.Float64, strict=False)
        agg.extend([
            vay.min().alias("min_vehicle_age_from_year"),
            vay.max().alias("max_vehicle_age_from_year"),
            vay.mean().alias("avg_vehicle_age_from_year"),
            vay.std().alias("std_vehicle_age_from_year"),
        ])

    for col in ["engine_power_clipped", "engine_volume_clipped"]:
        if col in df.columns:
            v = pl.col(col)
            agg.extend([
                v.min().alias(f"min_{col}"),
                v.max().alias(f"max_{col}"),
                v.mean().alias(f"avg_{col}"),
                v.std().alias(f"std_{col}"),
            ])
    for col in ["engine_power_outlier", "engine_volume_outlier",
                "engine_power_missing", "engine_volume_missing",
                "vehicle_age_is_new", "vehicle_age_is_old",
                "car_year_outlier", "car_year_missing"]:
        if col in df.columns:
            agg.append(pl.col(col).cast(pl.Int8).max().alias(f"any_{col}"))

    # --- 4.5 first-value categoricals + n_unique -----------------------------
    first_cat_cols = [
        "region_id", "region_name",
        "vehicle_type_id", "vehicle_type_name",
        "age_experience_id", "age_experience_name",
        "ownerkato_short",
        "car_age_category",
        "mark_clean", "model_clean",
        "mark_grouped", "model_grouped",
    ]
    for c in first_cat_cols:
        if c in df.columns:
            agg.append(pl.col(c).first().alias(f"{c}_first"))

    for c in ["mark", "model", "mark_grouped", "model_grouped", "region_id"]:
        if c in df.columns:
            agg.append(pl.col(c).n_unique().alias(f"{c}_n_unique"))

    # Date passthrough (constant within a policy, but use .first() to be safe).
    for c in ["operation_year", "operation_month",
              "operation_quarter", "operation_dayofweek",
              "operation_date_missing"]:
        if c in df.columns:
            agg.append(pl.col(c).first().alias(c))

    # --- 4.6 SCORE_* aggregation: mean + max only ----------------------------
    score_cols = [c for c in df.columns if c.startswith("SCORE_")]
    for c in score_cols:
        agg.append(pl.col(c).mean().alias(f"{c}_mean"))
        agg.append(pl.col(c).max().alias(f"{c}_max"))

    policy = df.group_by("contract_number").agg(agg)

    # --- 4.7 post-aggregation derived (still no target use) ------------------
    derived: list[pl.Expr] = []
    if {"max_experience_year", "min_experience_year"}.issubset(policy.columns):
        derived.append(
            (pl.col("max_experience_year") - pl.col("min_experience_year"))
            .alias("experience_range")
        )
    if {"max_vehicle_age_from_year", "min_vehicle_age_from_year"}.issubset(policy.columns):
        derived.append(
            (pl.col("max_vehicle_age_from_year") - pl.col("min_vehicle_age_from_year"))
            .alias("vehicle_age_range")
        )
    if {"n_unique_drivers", "n_unique_cars"}.issubset(policy.columns):
        derived.append(
            pl.when(pl.col("n_unique_cars") > 0)
              .then(pl.col("n_unique_drivers") / pl.col("n_unique_cars"))
              .otherwise(None).alias("drivers_per_car")
        )
        derived.append(
            pl.when(pl.col("n_unique_drivers") > 0)
              .then(pl.col("n_unique_cars") / pl.col("n_unique_drivers"))
              .otherwise(None).alias("cars_per_driver")
        )
    if derived:
        policy = policy.with_columns(derived)

    return policy


train_policy = aggregate_to_policy(train_clean, is_train=True)
test_policy  = aggregate_to_policy(test_clean,  is_train=False)
print(f"train_policy: {train_policy.shape}")
print(f"test_policy:  {test_policy.shape}")


## 5. Finalize

- Add a few safe flags from non-target features.
- Drop raw IDs / PII.
- Drop any accidental target columns in test.
- Enforce schema parity between train and test (except targets in train).


In [ ]:
def finalize(policy: pl.DataFrame, is_train: bool) -> pl.DataFrame:
    derived: list[pl.Expr] = []

    # --- 5.1 safe flags (non-target features only) ---------------------------
    if "n_unique_drivers" in policy.columns:
        derived.append((pl.col("n_unique_drivers") >= 3).alias("has_many_drivers"))
    if "n_unique_cars" in policy.columns:
        derived.append((pl.col("n_unique_cars") >= 2).alias("has_many_cars"))
    if "min_experience_year" in policy.columns:
        derived.append((pl.col("min_experience_year") <= 2).alias("has_low_experience_driver"))
    if "max_car_age_binary" in policy.columns:
        derived.append((pl.col("max_car_age_binary") == 1).alias("has_car_age_gt_7"))
    if "avg_vehicle_age_from_year" in policy.columns:
        derived.append((pl.col("avg_vehicle_age_from_year") >= 15).alias("has_old_vehicle"))

    # Premium per driver/car (uses premium_wo_term — NOT a target).
    if "premium_wo_term" in policy.columns:
        pwt = pl.col("premium_wo_term").cast(pl.Float64, strict=False)
        if "n_unique_drivers" in policy.columns:
            derived.append(
                pl.when(pl.col("n_unique_drivers") > 0)
                  .then(pwt / pl.col("n_unique_drivers"))
                  .otherwise(None).alias("premium_per_driver")
            )
        if "n_unique_cars" in policy.columns:
            derived.append(
                pl.when(pl.col("n_unique_cars") > 0)
                  .then(pwt / pl.col("n_unique_cars"))
                  .otherwise(None).alias("premium_per_car")
            )

    if derived:
        policy = policy.with_columns(derived)

    # --- 5.2 replace inf with null in numeric columns ------------------------
    numeric_cols = [c for c, dt in zip(policy.columns, policy.dtypes) if dt.is_numeric()]
    if numeric_cols:
        policy = policy.with_columns([
            pl.when(pl.col(c).is_infinite()).then(None).otherwise(pl.col(c)).alias(c)
            for c in numeric_cols
        ])

    # --- 5.3 fill string nulls with UNKNOWN (except contract_number) ---------
    string_cols = [c for c, dt in zip(policy.columns, policy.dtypes) if dt == pl.Utf8]
    fill_cols = [c for c in string_cols if c != "contract_number"]
    if fill_cols:
        policy = policy.with_columns([pl.col(c).fill_null("UNKNOWN") for c in fill_cols])

    # --- 5.4 drop raw IDs / PII ---------------------------------------------
    drop_cols = [c for c in RAW_DROP_COLS if c in policy.columns]
    if drop_cols:
        policy = policy.drop(drop_cols)

    # --- 5.5 on test, drop any accidental target columns ---------------------
    if not is_train:
        accidental = [c for c in TARGET_COLS if c in policy.columns]
        if accidental:
            policy = policy.drop(accidental)

    # --- 5.6 enforce unique contract_number ----------------------------------
    assert policy["contract_number"].n_unique() == policy.height, \
        "Duplicate contract_number in output!"

    return policy


train_final = finalize(train_policy, is_train=True)
test_final  = finalize(test_policy,  is_train=False)

print(f"train_final: {train_final.shape}")
print(f"test_final:  {test_final.shape}")


## 6. Schema alignment

Train must contain the targets `is_claim`, `claim_amount`, `claim_cnt`.
Test must contain the same feature columns as train, in the same order,
with the same dtypes — minus the targets.


In [ ]:
def align_schemas(train: pl.DataFrame, test: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    train_feats = [c for c in train.columns if c not in TARGET_COLS]
    test_feats  = list(test.columns)

    # Columns only in train (excluding targets) -> drop from train.
    train_only = [c for c in train_feats if c not in test_feats]
    if train_only:
        print(f"Dropping {len(train_only)} train-only feature cols: {train_only}")
        train = train.drop(train_only)

    # Columns only in test -> drop from test.
    test_only = [c for c in test_feats if c not in train.columns]
    if test_only:
        print(f"Dropping {len(test_only)} test-only feature cols: {test_only}")
        test = test.drop(test_only)

    # Reorder test to [contract_number, <features in train order>].
    feature_order = [c for c in train.columns if c not in TARGET_COLS]
    test = test.select(feature_order)

    # Reorder train to [contract_number, <features>, <targets at end>].
    target_order = [c for c in TARGET_COLS if c in train.columns]
    train = train.select(feature_order + target_order)

    # Align dtypes feature-by-feature: cast test feature to train feature dtype.
    train_dtypes = dict(zip(train.columns, train.dtypes))
    test_dtypes  = dict(zip(test.columns,  test.dtypes))
    cast_exprs = []
    for c in feature_order:
        if c == "contract_number":
            continue
        if test_dtypes.get(c) != train_dtypes.get(c):
            cast_exprs.append(pl.col(c).cast(train_dtypes[c], strict=False).alias(c))
    if cast_exprs:
        test = test.with_columns(cast_exprs)

    return train, test


train_final, test_final = align_schemas(train_final, test_final)

print(f"train_final: {train_final.shape}")
print(f"test_final:  {test_final.shape}")
print(f"train cols include targets: {[c for c in TARGET_COLS if c in train_final.columns]}")
print(f"test  cols include targets: {[c for c in TARGET_COLS if c in test_final.columns]}")


## 7. Leakage and sanity checks

In [ ]:
# --- 7.1 Forbidden feature-name substrings -----------------------------------
all_feature_cols = [c for c in train_final.columns if c not in TARGET_COLS]
bad = [
    c for c in all_feature_cols
    if any(sub in c for sub in FORBIDDEN_FEATURE_SUBSTRINGS)
]
assert not bad, f"Forbidden feature names found (leakage red flag): {bad}"
print("[OK] No forbidden feature-name substrings found.")

# --- 7.2 Correlation of numeric features with is_claim -----------------------
assert "is_claim" in train_final.columns, "is_claim missing from train_final"

numeric_feature_cols = [
    c for c, dt in zip(train_final.columns, train_final.dtypes)
    if c not in TARGET_COLS and c != "contract_number" and dt.is_numeric()
]
y = train_final["is_claim"].cast(pl.Float64).to_numpy()

high_corr: list[tuple[str, float]] = []
top_corrs: list[tuple[str, float]] = []
for c in numeric_feature_cols:
    x = train_final[c].cast(pl.Float64, strict=False).to_numpy()
    mask = np.isfinite(x) & np.isfinite(y)
    if mask.sum() < 10:
        continue
    xv, yv = x[mask], y[mask]
    if np.std(xv) == 0 or np.std(yv) == 0:
        continue
    r = float(np.corrcoef(xv, yv)[0, 1])
    if not np.isfinite(r):
        continue
    top_corrs.append((c, r))
    if abs(r) > 0.995:
        high_corr.append((c, r))

top_corrs.sort(key=lambda kv: abs(kv[1]), reverse=True)
print("Top 15 |corr| with is_claim:")
for c, r in top_corrs[:15]:
    print(f"  {c:50s}  corr={r:+.4f}")

assert not high_corr, (
    f"Possible target leakage: features with |corr|>0.995 vs is_claim: {high_corr}"
)
print("[OK] No numeric feature has |corr| > 0.995 with is_claim.")

# --- 7.3 Schema equality (features + dtypes) ---------------------------------
train_feat_schema = {
    c: dt for c, dt in zip(train_final.columns, train_final.dtypes)
    if c not in TARGET_COLS
}
test_feat_schema = dict(zip(test_final.columns, test_final.dtypes))

assert list(train_feat_schema.keys()) == list(test_feat_schema.keys()), (
    "Train/test feature column lists differ:\n"
    f"  only in train: {set(train_feat_schema) - set(test_feat_schema)}\n"
    f"  only in test:  {set(test_feat_schema) - set(train_feat_schema)}"
)
mismatched_dtypes = {
    c: (train_feat_schema[c], test_feat_schema[c])
    for c in train_feat_schema
    if train_feat_schema[c] != test_feat_schema[c]
}
assert not mismatched_dtypes, f"Train/test dtype mismatches: {mismatched_dtypes}"
print("[OK] Train and test share identical feature columns and dtypes.")

# --- 7.4 grain checks --------------------------------------------------------
assert train_final["contract_number"].n_unique() == train_final.height, \
    "Train contract_number is not unique"
assert test_final["contract_number"].n_unique() == test_final.height, \
    "Test contract_number is not unique"
print("[OK] contract_number is unique in both train and test.")

# --- 7.5 targets only in train ----------------------------------------------
assert all(c in train_final.columns for c in TARGET_COLS), \
    f"Missing targets in train: {[c for c in TARGET_COLS if c not in train_final.columns]}"
assert not any(c in test_final.columns for c in TARGET_COLS), \
    f"Targets leaked into test: {[c for c in TARGET_COLS if c in test_final.columns]}"
print("[OK] Targets present only in train.")


## 8. Write outputs

In [ ]:
train_path = PREP_DIR / "train.csv"
test_path  = PREP_DIR / "test.csv"

train_final.write_csv(train_path)
test_final.write_csv(test_path)

print(f"Wrote {train_path}  shape={train_final.shape}")
print(f"Wrote {test_path}   shape={test_final.shape}")
print(f"Feature count (excluding contract_number & targets): "
      f"{len([c for c in train_final.columns if c not in TARGET_COLS and c != 'contract_number'])}")
